In [1]:
import os
import torch
#from utils.rot6d import robust_compute_rotation_matrix_from_ortho6d
import pytorch_kinematics as pk
import trimesh

In [62]:
def normalize_vector(v):
    batch = v.shape[0]
    v_mag = torch.sqrt(v.pow(2).sum(1))  # batch
    v_mag = torch.max(v_mag, v.new([1e-8]))
    v_mag = v_mag.view(batch, 1).expand(batch, v.shape[1])
    v = v/v_mag
    return v

def cross_product(u, v):
    batch = u.shape[0]
    i = u[:, 1] * v[:, 2] - u[:, 2] * v[:, 1]
    j = u[:, 2] * v[:, 0] - u[:, 0] * v[:, 2]
    k = u[:, 0] * v[:, 1] - u[:, 1] * v[:, 0]
        
    out = torch.cat((i.view(batch, 1), j.view(batch, 1), k.view(batch, 1)), 1)
        
    return out

def robust_compute_rotation_matrix_from_ortho6d(poses):
    """
    Instead of making 2nd vector orthogonal to first
    create a base that takes into account the two predicted
    directions equally
    """
    x_raw = poses[:, 0:3]  # batch*3
    y_raw = poses[:, 3:6]  # batch*3

    x = normalize_vector(x_raw)  # batch*3
    y = normalize_vector(y_raw)  # batch*3
    middle = normalize_vector(x + y)
    orthmid = normalize_vector(x - y)
    x = normalize_vector(middle + orthmid)
    y = normalize_vector(middle - orthmid)
    # Their scalar product should be small !
    # assert torch.einsum("ij,ij->i", [x, y]).abs().max() < 0.00001
    z = normalize_vector(cross_product(x, y))

    x = x.view(-1, 3, 1)
    y = y.view(-1, 3, 1)
    z = z.view(-1, 3, 1)
    matrix = torch.cat((x, y, z), 2)  # batch*3*3
    # Check for reflection in matrix ! If found, flip last vector TODO
    # assert (torch.stack([torch.det(mat) for mat in matrix ])< 0).sum() == 0
    return matrix

In [55]:
path = 'leaphand.urdf'
chain = pk.build_chain_from_urdf(
    open(path).read()
).to(dtype=torch.float)

Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='0']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='1']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='2']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='3']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='4']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='5']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='6']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='7']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='8']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='9']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='10']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='11']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='12']
Unknown tag "joint_pro

In [14]:
body = chain._root
device = torch.device("cpu")
if (len(body.link.visuals) > 0):
    link_vertices = []
    link_faces = []
    n_link_vertices = 0
    for visual in body.link.visuals:
        print(visual.geom_param)
        print(visual.offset.to(dtype=torch.float, device=device))

palm_lower.stl
Transform3d(rot=tensor([[1., 0., 0., 0.]]), pos=tensor([[-0.0201,  0.0258, -0.0347]]))


In [131]:
class shit:
    def __init__(self, path, device) -> None:
        self.device = device
        self.chain = chain = pk.build_chain_from_urdf(
            open(path).read()).to(dtype=torch.float)
        self.count=0
        self.mesh = None
        self.meshes = {}
        def get_mesh(body):
            if (len(body.link.visuals) > 0):
                link_vertices = []
                link_faces = []
                n_link_vertices = 0

                for visual in body.link.visuals:
                    link_mesh = trimesh.load_mesh(visual.geom_param, file_type='stl', process=False)
                    self.count+=1
                    vertices = torch.tensor(
                            link_mesh.vertices, dtype=torch.float)
                    faces = torch.tensor(
                            link_mesh.faces, dtype=torch.float)
                    pos = visual.offset.to(dtype=torch.float, device=self.device)
                    '''print(body.link.name)
                    print(pos)'''
                    vertices = pos.transform_points(vertices)
                    link_vertices.append(vertices)
                    link_faces.append(faces + n_link_vertices)
                    n_link_vertices += len(vertices)
                link_vertices = torch.cat(link_vertices, dim=0)
                link_faces = torch.cat(link_faces, dim=0)
                self.meshes[body.link.name] = {'vertices': link_vertices,
                                                'faces': link_faces,
                                            }
            for children in body.children:
                get_mesh(children)
        
        get_mesh(self.chain._root)
    
    def set_parameters(self, hand_pose):
        """
        Set translation, rotation, and joint angles of grasps
        
        Parameters
        ----------
        hand_pose: (B, 3+6+`n_dofs`) torch.FloatTensor
            translation, rotation in rot6d, and joint angles
        """
        self.hand_pose = hand_pose
        if self.hand_pose.requires_grad:
            self.hand_pose.retain_grad()
        self.global_translation = self.hand_pose[:, 0:3]
        self.global_rotation = robust_compute_rotation_matrix_from_ortho6d(
            self.hand_pose[:, 3:9])
        self.current_status = self.chain.forward_kinematics(
            self.hand_pose[:, 9:])

    def get_trimesh_data(self, i):
        """
        Get full mesh
        
        Returns
        -------
        data: trimesh.Trimesh
        """
        data = trimesh.Trimesh()
        for link_name in self.meshes:
            v = self.current_status[link_name].transform_points(
                self.meshes[link_name]['vertices'])
            if len(v.shape) == 3:
                v = v[i]
            v = v @ self.global_rotation[i].T + self.global_translation[i]
            v = v.detach().cpu()
            f = self.meshes[link_name]['faces'].detach().cpu()
            data += trimesh.Trimesh(vertices=v, faces=f)
        return data
        


In [132]:
Shit = shit(path, device)

Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='0']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='1']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='2']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='3']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='4']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='5']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='6']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='7']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='8']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='9']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='10']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='11']
Unknown tag "joint_properties" in /robot[@name='onshape']/joint[@name='12']
Unknown tag "joint_pro

In [120]:
pos = torch.tensor([-1.5736e-05, -2.0027e-05,  5.9941e-02,  1.0000e+00,  6.6959e-04,
        -7.5518e-04,  7.1168e-04,  6.2748e-02,  9.9803e-01, -6.3991e-02,
         1.3352e-01,  3.2812e-01,  1.7646e-01,  1.5890e-02,  8.1746e-02,
         1.7309e-01,  4.6221e-02, -5.0946e-02,  2.3177e-01,  1.2505e-01,
         1.2294e-01,  3.1164e-02,  1.3962e-02,  2.1124e-01,  1.0541e-01])
Shit.set_parameters(pos.unsqueeze(0))

In [133]:
mesh = Shit.get_trimesh_data(0)

In [134]:
mesh.vertices

TrackedArray([[-0.06409524,  0.00044802, -0.01978039],
              [-0.06309525,  0.00042736, -0.01976164],
              [-0.06309525,  0.00085207, -0.02025717],
              ...,
              [ 0.00978978, -0.02467236, -0.0286    ],
              [ 0.0098    , -0.02475   , -0.0286    ],
              [ 0.0098    , -0.0265    , -0.0286    ]])

In [135]:
mesh.apply_scale(100)
mesh.show()